# ULTRON on Google Colab — phone-ready

Run the full Ultron app on Colab's hardware and open it from your **phone's browser**.

## Just want it running? Run the ONE cell below.

It clones the repo, installs deps, asks for your [Groq key](https://console.groq.com/keys), starts the server, and prints a **public `trycloudflare.com` link** (no account, no token) that opens on any phone. That's it.

The cells further down are the same steps broken apart, plus optional extras (ngrok, Google Drive memory, GPU voice). You don't need them unless you want them.

*Notes: everything runs on a throwaway Colab VM, not your device. The DB and any files vanish when the runtime resets — use the Drive cell to keep memory. Every cell here is safe to re-run.*

## ▶️ One-shot: clone → install → key → launch

In [ ]:
import os, subprocess, sys, time, threading

REPO = "https://github.com/Razoradams9/ultron.git"
DEST = "/content/ultron"

# 0. escape a dead cwd. If a previous run deleted the folder this runtime
#    was sitting in, every command errors with 'getcwd: cannot access...'.
#    Standing in /content (always present) makes the rest of the cell work.
os.makedirs("/content", exist_ok=True)
os.chdir("/content")

# 1. clone or update (never destructive)
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO, DEST], check=True)
os.chdir(DEST)

# 2. deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"], check=True)

# 3. Groq key (only asks once per session)
if not os.environ.get("GROQ_API_KEY"):
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ").strip()
os.environ["ULTRON_PROVIDER"] = "groq"

# 4. launch (idempotent — frees the port if a previous run is still up)
subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
time.sleep(1)
import nest_asyncio, uvicorn
nest_asyncio.apply()

def _serve():
    uvicorn.run("ultron.server:app", host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_serve, daemon=True).start()

# wait until the server actually answers before exposing it (avoids 404s
# from opening the tunnel before uvicorn has bound the port)
import urllib.request
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=2)
        break
    except Exception:
        time.sleep(0.5)

# 5. public link via Cloudflare quick tunnel — NO account, NO token, works on
#    any phone/network. This is more reliable than Colab's proxy, which is
#    tied to your desktop Google session and often 404s on other devices.
import re
url = None
try:
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "&& chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True,
    )
    subprocess.run("pkill -f cloudflared 2>/dev/null", shell=True); time.sleep(1)
    logf = "/content/cloudflared.log"
    open(logf, "w").close()
    subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
        stdout=open(logf, "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        time.sleep(1)
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(logf).read())
        if m:
            url = m.group(0)
            break
except Exception as e:
    print("cloudflare tunnel unavailable (", e, ")")

if url:
    print("\n🔴  ULTRON is live — open this on your phone (works anywhere):\n\n    " + url + "\n")
else:
    # last resort: Colab's own proxy (desktop-session only — may 404 on phone)
    from google.colab.output import eval_js
    proxy = eval_js("google.colab.kernel.proxyPort(8000)")
    print("\n🔴  ULTRON (Colab proxy — may not work on phone):\n\n    " + proxy + "\n")
if not os.environ.get("GROQ_API_KEY"):
    print("⚠️  No Groq key set — the dashboard loads but the brain won't reply.")

---

## Step-by-step (optional — skip if the one-shot cell worked)

### 1. Clone / update the repo

In [ ]:
import os
REPO, DEST = "https://github.com/Razoradams9/ultron.git", "/content/ultron"
os.makedirs("/content", exist_ok=True)
%cd /content
if os.path.isdir(os.path.join(DEST, ".git")):
    %cd $DEST
    !git pull --ff-only
else:
    !git clone $REPO $DEST
    %cd $DEST
!ls

### 2. Install dependencies

In [ ]:
%cd /content/ultron
!pip install -q -r requirements.txt nest_asyncio
print("deps installed")

### 3. Set your Groq API key

From [console.groq.com/keys](https://console.groq.com/keys). Lives only in this session. Uncomment the persona line for the polite butler.

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ").strip()
os.environ["ULTRON_PROVIDER"] = "groq"
# os.environ["ULTRON_PERSONA"] = "jarvis"
print("Key set." if os.environ.get("GROQ_API_KEY") else "No key entered.")

### 4. Launch + open the dashboard

Safe to re-run — frees port 8000 first, waits for the server to answer, then opens a Cloudflare quick tunnel (no account/token) that works on any phone.

In [ ]:
import os, re, subprocess, time, threading, urllib.request
os.chdir("/content/ultron")
subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True); time.sleep(1)
import nest_asyncio, uvicorn
nest_asyncio.apply()
def _serve():
    uvicorn.run("ultron.server:app", host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_serve, daemon=True).start()

# wait for the server to answer before exposing it
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=2); break
    except Exception:
        time.sleep(0.5)

# Cloudflare quick tunnel — no account/token, works on any phone
subprocess.run(
    "wget -q -O /usr/local/bin/cloudflared "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "&& chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
subprocess.run("pkill -f cloudflared 2>/dev/null", shell=True); time.sleep(1)
logf = "/content/cloudflared.log"; open(logf, "w").close()
subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                 stdout=open(logf, "a"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(logf).read())
    if m: url = m.group(0); break
if url:
    print("\n🔴  ULTRON:", url)
else:
    from google.colab.output import eval_js
    print("\n🔴  ULTRON (Colab proxy):", eval_js("google.colab.kernel.proxyPort(8000)"))

### 5. Health check (optional)

In [ ]:
import urllib.request, json
try:
    r = urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=8)
    print("OK:", json.dumps(json.loads(r.read().decode()), indent=2))
except Exception as e:
    print("Server not up yet:", e, "— re-run the launch cell.")

---

## Extras (all optional)

### A stable ngrok link (nicer on mobile than the Colab proxy)

Add a free [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken), run this cell, then re-run the launch cell above — it'll print an ngrok URL instead.

In [ ]:
from getpass import getpass
token = getpass("ngrok authtoken (Enter to skip): ").strip()
if token:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    ngrok.kill()
    print("\n🔴  ULTRON (ngrok):", ngrok.connect(8000).public_url)
else:
    print("Skipped — keep using the Colab proxy link.")

### Persist memory to Google Drive

Run **before** launching so `ultron.db` survives runtime resets.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/ultron", exist_ok=True)
db, link = "/content/drive/MyDrive/ultron/ultron.db", "/content/ultron/ultron.db"
if not os.path.islink(link):
    if os.path.exists(link):
        os.remove(link)
    os.symlink(db, link)
print("Memory persisted at", db)

### GPU-accelerated cloned voice (heavy — only if you want spoken replies)

The app works fine without this; spoken replies just won't be available.

1. Set the runtime to GPU: *Runtime → Change runtime type → T4 GPU*, then re-run the setup cells.
2. Upload a 10–60s voice clip via the folder icon on the left (optional — skip for the stock voice).
3. Run the cell below, then **re-run the launch cell** so the app picks up `VOICE_URL`.

First synthesis takes ~30s while the model loads; after that it's cached.

In [ ]:
import os, glob, time, threading, subprocess

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then re-run setup cells.")
os.environ["VOICE_DEVICE"] = "cuda"

# voice deps, installed without disturbing Colab's CUDA torch
!pip install -q --no-deps chatterbox-tts
!pip install -q pykakasi==2.3.0 pyloudnorm resemble-perth s3tokenizer spacy-pkuseg conformer librosa soundfile

# register the newest uploaded clip as the voiceprint (optional)
clips = [f for f in glob.glob("/content/*")
         if f.lower().endswith((".mp3", ".wav", ".m4a", ".mpeg", ".flac", ".ogg"))]
clips.sort(key=os.path.getmtime, reverse=True)
if clips:
    import librosa, soundfile as sf
    os.makedirs("/content/ultron/voice/samples", exist_ok=True)
    y, sr = librosa.load(clips[0], sr=24000, mono=True)
    sf.write("/content/ultron/voice/samples/reference.wav", y, 24000, subtype="PCM_16")
    print("voiceprint from:", clips[0], f"({len(y)/sr:.0f}s)")
else:
    print("no clip uploaded — using the stock voice")

# start the voice service on :8001 (idempotent)
os.chdir("/content/ultron")
subprocess.run("fuser -k 8001/tcp 2>/dev/null", shell=True); time.sleep(1)
def _voice():
    import uvicorn
    uvicorn.run("voice.server:app", host="127.0.0.1", port=8001, log_level="warning")
threading.Thread(target=_voice, daemon=True).start()
time.sleep(5)
os.environ["VOICE_URL"] = "http://127.0.0.1:8001"
print("voice service on :8001 — now RE-RUN the launch cell so the app connects to it.")